In [2]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

prepare the data

In [3]:
df = fetch_california_housing(as_frame=True).frame.sample(300, random_state=42)
X_raw = df[['MedInc']].values
y = df[['MedHouseVal']].values

In [4]:
# 2. عمل Polynomial وتجهيز المدخلات
poly = PolynomialFeatures(degree=2)
X = poly.fit_transform(X_raw)
X = StandardScaler().fit_transform(X)
X[:, 0] = 1.0  # Bias column

m, n = X.shape
lr = 0.05
epochs = 50

In [5]:
# دالة بسيطة لطباعة النتيجة
def print_metrics(name, w):
  preds = X @ w
  loss = np.mean((preds - y) ** 2)
  acc = r2_score(y, preds) * 100
  print(f"[{name}] -> Loss: {loss:.4f} | Accuracy: {acc:.2f}%")

Gradient Descent

In [6]:
w = np.zeros((n, 1))

for epoch in range(epochs):
  # حساب الجرادينت على كل الداتا دفعة واحدة
  grad = (2 / m) * X.T @ (X @ w - y)
  w -= lr * grad

print_metrics("Gradient Descent", w)

[Gradient Descent] -> Loss: 0.7118 | Accuracy: 44.07%


Stochastic Gradient Descent (SGD)

In [7]:
w = np.zeros((n, 1))

for epoch in range(epochs):
  for i in range(m):
    # عينة واحدة في كل خطوة
    xi = X[i : i + 1]
    yi = y[i : i + 1]
    grad = 2 * xi.T @ (xi @ w - yi)
    w -= lr * grad

print_metrics("SGD", w)

[SGD] -> Loss: 0.7558 | Accuracy: 40.61%


Mini-batch SGD

In [8]:
w = np.zeros((n, 1))
batch_size = 16

for epoch in range(epochs):
  for i in range(0, m, batch_size):
    xb = X[i : i + batch_size]
    yb = y[i : i + batch_size]
    grad = (2 / len(xb)) * xb.T @ (xb @ w - yb)
    w -= lr * grad

print_metrics("Mini-batch SGD", w)

[Mini-batch SGD] -> Loss: 0.6742 | Accuracy: 47.02%


AdaGrad

In [9]:
w = np.zeros((n, 1))
s = np.zeros((n, 1))  # تراكم مربعات الجرادينت
eps = 1e-8
batch_size = 16

for epoch in range(epochs):
  for i in range(0, m, batch_size):
    xb = X[i : i + batch_size]
    yb = y[i : i + batch_size]
    grad = (2 / len(xb)) * xb.T @ (xb @ w - yb)

    s += grad**2
    w -= (lr / (np.sqrt(s) + eps)) * grad

print_metrics("AdaGrad", w)

[AdaGrad] -> Loss: 0.7250 | Accuracy: 43.03%


Adam

In [10]:
w = np.zeros((n, 1))
v = np.zeros((n, 1))  # momentum
s = np.zeros((n, 1))  # scaling
beta1, beta2, eps = 0.9, 0.999, 1e-8
t = 0
batch_size = 16

for epoch in range(epochs):
  for i in range(0, m, batch_size):
    t += 1
    xb = X[i : i + batch_size]
    yb = y[i : i + batch_size]
    grad = (2 / len(xb)) * xb.T @ (xb @ w - yb)

    # تحديث العزوم
    v = beta1 * v + (1 - beta1) * grad
    s = beta2 * s + (1 - beta2) * (grad**2)

    # تصحيح الانحياز (Bias correction)
    v_corr = v / (1 - beta1**t)
    s_corr = s / (1 - beta2**t)

    w -= (lr / (np.sqrt(s_corr) + eps)) * v_corr

print_metrics("Adam", w)

[Adam] -> Loss: 0.6742 | Accuracy: 47.02%
